In [8]:
import torch
from torch.optim import AdamW
"""
AdamW 是 PyTorch 内置的优化器，Adam 的改进版。
它的改进是把**权重衰减（weight decay）**从 Adam 的 L2 正则化里拆出来，直接做解耦衰减。这样学习率和权重衰减各管各的，不会互相干扰。
"""
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 和之前一样
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# 新增部分
batch["labels"] = torch.tensor([1, 1])
## 手动构造两个标签，比如 [1, 1] 表示两个样本的真实类别都是第 1 类（如 POSITIVE）
optimizer = AdamW(model.parameters())
# 创建 AdamW 优化器，把模型所有可训练参数交给它管理
loss = model(**batch).loss
# 前向传播：batch 含 input_ids、attention_mask、labels，model 内部自动算 loss
loss.backward()
# 反向传播：计算每个参数的梯度
optimizer.step()
# 根据梯度更新参数（学习率 × 梯度方向）

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
#加载数据集
from datasets import load_dataset

raw_datasets = load_dataset("glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

现在我们获得了一个 DatasetDict 对象，这个对象包含训练集、验证集和测试集。每一个集合都包含 4 个列（ sentence1 ， sentence2 ， label 和 idx ）以及一个代表行数的变量（每个集合中的行的个数）。运行结果显示该训练集中有 3668 对句子，验证集中有 408 对，测试集中有 1725 对。

默认情况下，该命令会下载数据集并缓存到 ~/.cache/huggingface/datasets 。HF_HOME 环境变量来自定义缓存的文件夹。

我们可以访问该数据集中的每一个 raw_train_dataset 对象，例如使用字典：

In [10]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

现在可以看到标签已经是整数了，因此不需要对标签做任何预处理。（如果原始标签是 "positive"/"negative" 这类字符串，就得先手动映射成 0/1。）如果想要知道不同数字对应标签的实际含义，我们可以查看 raw_train_dataset 的 features 。这告诉我们每列的类型：

In [11]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

Label（标签） 是一种 ClassLabel（分类标签） ，也就是使用整数建立起类别标签的映射关系。 0 对应于 not_equivalent（非同义） ， 1 对应于 equivalent（同义） 。

In [12]:
#查看训练集的第 15 行元素和验证集的 87 行元素。他们的标签
raw_train_dataset[15], raw_datasets["validation"][87]

({'sentence1': 'Rudder was most recently senior vice president for the Developer & Platform Evangelism Business .',
  'sentence2': 'Senior Vice President Eric Rudder , formerly head of the Developer and Platform Evangelism unit , will lead the new entity .',
  'label': 0,
  'idx': 16},
 {'sentence1': 'However , EPA officials would not confirm the 20 percent figure .',
  'sentence2': 'Only in the past few weeks have officials settled on the 20 percent figure .',
  'label': 0,
  'idx': 812})

In [14]:
inputs = tokenizer("This is the first sentence.", "This is the second one.")
inputs

{'input_ids': [101, 2023, 2003, 1996, 2034, 6251, 1012, 102, 2023, 2003, 1996, 2117, 2028, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

token类型ID(token_type_ids) 的作用就是告诉模型输入的哪一部分是第一句，哪一部分是第二句。

In [16]:
#选取训练集中的第 15 个元素，将两句话分别进行tokenization。
#结果和上方的例子有什么不同？
tokenized_sentences_15 = tokenizer(raw_train_dataset[15]["sentence1"], raw_train_dataset[15]["sentence2"])
tokenized_sentences_15

{'input_ids': [101, 24049, 2001, 2087, 3728, 3026, 3580, 2343, 2005, 1996, 9722, 1004, 4132, 9340, 12439, 2964, 2449, 1012, 102, 3026, 3580, 2343, 4388, 24049, 1010, 3839, 2132, 1997, 1996, 9722, 1998, 4132, 9340, 12439, 2964, 3131, 1010, 2097, 2599, 1996, 2047, 9178, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
#将 input_ids 中的 id 转换回文字：
tokenizer.convert_ids_to_tokens(inputs["input_ids"])
#模型需要输入的形式是 [CLS] sentence1 [SEP] sentence2 [SEP] 。

['[CLS]',
 'this',
 'is',
 'the',
 'first',
 'sentence',
 '.',
 '[SEP]',
 'this',
 'is',
 'the',
 'second',
 'one',
 '.',
 '[SEP]']

输入中 [CLS] sentence1 [SEP] 它们的 token_type_ids 均为 0 ，而其他部分例如 sentence2 [SEP] ，所有的 token_type_ids 均为 1 。

如果选择其他的 checkpoint，不一定具有 token_type_ids ，比如，DistilBERT 模型就不会返回。只有当 tokenizer 在预训练期间使用过这一层，也就是模型在构建时需要它们时，才会返回 token_type_ids 。

在这里，BERT 使用了带有 token_type_ids 的预训练 tokenizer，掩码语言建模，还有一个额外的应用类型称为“下一句预测”。这个任务的目标是对句子对之间的关系进行建模。

在下一句预测任务中，会给模型输入成对的句子（带有随机遮罩的 token），并要求预测第二个句子是否紧跟第一个句子。为了使任务具有挑战性，提高模型的泛化能力，数据集中一有一半句子对中的句子在原始文档中顺序排列，另一半句子对中的两个句子来自两个不同的文档。

一般来说无需要担心在你的输入中是否需要有 token_type_ids 。只要你使用相同的 checkpoint 的 Tokenizer 和模型，Tokenizer 就会知道向模型提供什么，一切都会顺利进行。

预处理训练数据集的一种方法是：
tokenized_dataset = tokenizer(
    raw_datasets["train"]["sentence1"],
    raw_datasets["train"]["sentence2"],
    padding=True,
    truncation=True,
)

这种方法虽然有效，但有一个缺点是它返回的是一个字典（字典的键是 输入词id(input_ids) ， 注意力遮罩(attention_mask) 和 token类型ID(token_type_ids) ，字典的值是键所对应值的列表）。这意味着在转换过程中要有足够的内存来存储整个数据集才不会出错。不过来自Datasets 库中的数据集是以 Apache Arrow 格式存储在磁盘上的，因此你只需将接下来要用的数据加载在内存中，而不是加载整个数据集，这对内存容量的需求比较友好。

我们将使用 Dataset.map() 方法将数据保存为 dataset 格式，如果我们需要做更多的预处理而不仅仅是 tokenization 它还支持了一些额外的自定义的方法。 map() 方法的工作原理是使用一个函数处理数据集的每个元素。让我们定义一个对输入进行 tokenize 的函数：
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
该函数接收一个字典（与 dataset 的项类似）并返回一个包含 输入词id(input_ids) ， 注意力遮罩(attention_mask) 和 token_type_ids 键的新字典。请注意，如果 example 字典所对应的值包含多个句子（每个键作为一个句子列表），那么它依然可以运行，就像前面的例子一样， tokenizer 可以处理成对的句子列表，这样的话我们可以在调用 map() 时使用该选项 batched=True ，这将显著加快处理的速度。 tokenizer 来自Tokenizers 库，由 Rust 编写而成。当一次给它很多输入时，这个 tokenizer 可以处理地非常快。

请注意，我们暂时在 tokenize_function 中省略了 padding 参数。这是因为将所有的样本填充到最大长度有些浪费。一个更好的做法是：在构建 batch 的时候。这样我们只需要填充到每个 batch 中的最大长度，而不是整个数据集的最大长度。当输入长度不稳定时，这可以节省大量时间和处理能力！

下面是我们如何使用一次性 tokenize_function 处理整个数据集。我们在调用 map 时使用了 batched =True ，这样函数就可以同时处理数据集的多个元素，而不是分别处理每个元素，这样可以更快进行预处理。

batched=True 表示一次传入一批（默认 1000 条），兼顾速度和内存。处理完返回的还是 Dataset，后续直接扔给 Trainer。

In [19]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

在使用预处理函数 map() 时，甚至可以通过传递 num_proc 参数并行处理。我们在这里没有这样做，因为在这个例子中Tokenizers 库已经使用多线程来更快地对样本 tokenize，但是如果没有使用该库支持的快速 tokenizer，使用 num_proc 可能会加快预处理。

tokenize_function 返回包含 输入词id(input_ids) ， 注意力遮罩(attention_mask) 和 token_type_ids 键的字典，这三个字段被添加到数据集的三个集合里（训练集、验证集和测试集）。请注意，如果预处理函数 map() 为现有键返回一个新值，我们可以通过使用 map() 函数返回的新值修改现有的字段。

我们最后需要做的是将所有示例填充到该 batch 中最长元素的长度，这种技术被称为动态填充。

In [20]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [21]:
#为了测试这个新东西，让我们从我们的训练集中抽取几个样本，
# 在这里，我们删除列 idx ， sentence1 和 sentence2 ，
# 因为不需要它们，而且删除包含字符串的列（我们不能用字符串创建张量），
# 然后查看一个 batch 中每个条目的长度：
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items() if k not in ["idx", "sentence1", "sentence2"]}
[len(x) for x in samples["input_ids"]]

[50, 59, 47, 67, 59, 50, 62, 32]

In [22]:
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

{'input_ids': torch.Size([8, 67]),
 'token_type_ids': torch.Size([8, 67]),
 'attention_mask': torch.Size([8, 67]),
 'labels': torch.Size([8])}